# AI 1 - Lab 3: Cluster Analysis

**Names:** Avery John, Josh Samuel, Zachary Jennings  
**Dataset:** UCI Secondary Mushroom Dataset  
**Method:** Cluster Analysis (Option A in the Modeling & Evaluation rubric)

This lab builds on our MiniLab and Lab 2. We use the same cleaned mushroom dataset, but this time we are looking for natural groups of mushrooms instead of predicting a class.

**Research question:** Can we find meaningful clusters based on mushroom characteristics, and what do those clusters tell us about the data?

**Note:** The assignment calls clustering *Option B* on page 1 and *Option A* in the grading rubric. We are following **Option A: Cluster Analysis** from the rubric.

## 1. Business Understanding (CRISP-DM)

The UCI Secondary Mushroom Dataset contains physical characteristics of mushrooms, along with edible or poisonous labels. In our previous labs, we used these characteristics for classification. Here, we want to see whether similar mushrooms naturally form groups without using the labels during modeling.

Clustering could help researchers explore patterns in shape, color, size, and habitat. It could also suggest new variables for a future classification model. The clusters do **not** automatically represent different mushroom species or safety categories.

**What we want to accomplish:**

1. Compare K-Means, Agglomerative Clustering, and DBSCAN.
2. Find a reasonable number of clusters (or suitable density settings for DBSCAN).
3. Evaluate how well the groups are separated using silhouette score, Davies-Bouldin index, and cluster sizes.
4. Describe the common characteristics of each group and compare them with the known edible/poisonous labels **after** clustering.

We will mainly judge the methods by their internal clustering scores and whether their clusters are understandable. We will also use Adjusted Rand Index (ARI) as a post-modeling check against the known labels. We will not use those labels to choose the clustering parameters.

**Safety limitation:** These are exploratory groups. They should never be used to decide whether a wild mushroom is safe to eat.

## 2. Data Understanding

### 2.1 Load Libraries

We use pandas and matplotlib for exploring the data and scikit-learn for preprocessing, clustering, and evaluation. The code is similar to the pipelines we used in Lab 2.

In [ ]:
# Load libraries
import os
import time
import certifi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score)

pd.set_option("display.max_columns", 50)
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
RANDOM_STATE = 42

### 2.2 Load and Clean the Mushroom Data

We reuse our cleaning functions from Lab 2 so the dataset stays consistent with our earlier work. We decode categorical values, remove exact duplicate rows, fill missing categorical values with `Missing`, and create our two previous engineered variables.

The cleaning code is included below so this notebook can be run by itself.

In [ ]:
# clean.py
import os
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())

import pandas as pd
from ucimlrepo import fetch_ucirepo

decode_map = {
    'class': {'p': 'poisonous', 'e': 'edible'},
    'cap-shape': {'b': 'bell', 'c': 'conical', 'x': 'convex', 'f': 'flat',
                  's': 'sunken', 'p': 'spherical', 'o': 'others'},
    'cap-surface': {'i': 'fibrous', 'g': 'grooves', 'y': 'scaly', 's': 'smooth',
                    'h': 'shiny', 'l': 'leathery', 'k': 'silky', 't': 'sticky',
                    'w': 'wrinkled', 'e': 'fleshy'},
    'cap-color': {'n': 'brown', 'b': 'buff', 'g': 'gray', 'r': 'green', 'p': 'pink',
                  'u': 'purple', 'e': 'red', 'w': 'white', 'y': 'yellow',
                  'l': 'blue', 'o': 'orange', 'k': 'black'},
    'does-bruise-or-bleed': {'t': 'yes', 'f': 'no'},
    'gill-attachment': {'a': 'adnate', 'x': 'adnexed', 'd': 'decurrent', 'e': 'free',
                        's': 'sinuate', 'p': 'pores', 'f': 'none', '?': 'unknown'},
    'gill-spacing': {'c': 'close', 'd': 'distant', 'f': 'none'},
    'gill-color': {'n': 'brown', 'b': 'buff', 'g': 'gray', 'r': 'green', 'p': 'pink',
                   'u': 'purple', 'e': 'red', 'w': 'white', 'y': 'yellow',
                   'l': 'blue', 'o': 'orange', 'k': 'black', 'f': 'none'},
    'stem-root': {'b': 'bulbous', 's': 'swollen', 'c': 'club', 'u': 'cup',
                  'e': 'equal', 'z': 'rhizomorphs', 'r': 'rooted', 'f': 'fibrous'},
    'stem-surface': {'i': 'fibrous', 'g': 'grooves', 'y': 'scaly', 's': 'smooth',
                     'h': 'shiny', 'l': 'leathery', 'k': 'silky', 't': 'sticky',
                     'w': 'wrinkled', 'e': 'fleshy', 'f': 'none'},
    'stem-color': {'n': 'brown', 'b': 'buff', 'g': 'gray', 'r': 'green', 'p': 'pink',
                   'u': 'purple', 'e': 'red', 'w': 'white', 'y': 'yellow',
                   'l': 'blue', 'o': 'orange', 'k': 'black', 'f': 'none'},
    'veil-type': {'p': 'partial', 'u': 'universal'},
    'veil-color': {'n': 'brown', 'b': 'buff', 'g': 'gray', 'r': 'green', 'p': 'pink',
                   'u': 'purple', 'e': 'red', 'w': 'white', 'y': 'yellow',
                   'l': 'blue', 'o': 'orange', 'k': 'black', 'f': 'none'},
    'has-ring': {'t': 'ring', 'f': 'none'},
    'ring-type': {'c': 'cobwebby', 'e': 'evanescent', 'r': 'flaring', 'g': 'grooved',
                  'l': 'large', 'p': 'pendant', 's': 'sheathing', 'z': 'zone',
                  'y': 'scaly', 'm': 'movable', 'f': 'none', '?': 'unknown'},
    'spore-print-color': {'n': 'brown', 'b': 'buff', 'g': 'gray', 'r': 'green', 'p': 'pink',
                          'u': 'purple', 'e': 'red', 'w': 'white', 'y': 'yellow',
                          'l': 'blue', 'o': 'orange', 'k': 'black'},
    'season': {'s': 'spring', 'u': 'summer', 'a': 'autumn', 'w': 'winter'},
    'habitat': {'g': 'grasses', 'l': 'leaves', 'm': 'meadows', 'p': 'paths',
                'h': 'heaths', 'u': 'urban', 'w': 'waste', 'd': 'woods'}
}


def load_raw() -> pd.DataFrame:
    secondary_mushroom = fetch_ucirepo(id=848)
    X = secondary_mushroom.data.features
    y = secondary_mushroom.data.targets
    return pd.concat([X, y], axis=1)


def clean_mushrooms(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Decode letter codes to words
    for col, mapping in decode_map.items():
        df[col] = df[col].map(mapping)

    # Drop exact duplicates (146 rows, -> 60,923)
    df = df.drop_duplicates().reset_index(drop=True)

    # Fill missing categoricals with an explicit "Missing" category
    cat_cols = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
    df[cat_cols] = df[cat_cols].fillna("Missing")

    # Engineered features
    df["has-stem"] = ~((df["stem-height"] == 0) & (df["stem-width"] == 0))
    df["has-stem"] = df["has-stem"].map({True: "yes", False: "no"})
    df["size-score"] = df["cap-diameter"] + df["stem-height"] + df["stem-width"] / 10

    return df

In [ ]:
# Load the raw and cleaned data
raw_df = load_raw()
df = clean_mushrooms(raw_df)

print("Raw data:", raw_df.shape)
print("Clean data:", df.shape)
print("Exact duplicates removed:", raw_df.duplicated().sum())
print("Missing values after cleaning:", df.isna().sum().sum())

display(df.head())

### 2.3 Attribute Descriptions and Data Quality

Most variables describe mushroom appearance or the environment where the mushroom was found. Measurements such as cap diameter and stem width are numerical. Variables such as cap shape and habitat are categorical. `class` is the known edible/poisonous label from Lab 2.

The table below lists **every attribute**, its meaning, and its observed values or range. `has-stem` and `size-score` were added in our cleaning step rather than supplied as original fields.

In [ ]:
# Describe all attributes (including variables created in Lab 2)
descriptions = {
    "cap-diameter": "Cap diameter", "cap-shape": "Shape of the cap",
    "cap-surface": "Surface texture of the cap", "cap-color": "Color of the cap",
    "does-bruise-or-bleed": "Whether it bruises or bleeds", 
    "gill-attachment": "How the gills attach", "gill-spacing": "Spacing between gills",
    "gill-color": "Color of the gills", "stem-height": "Height of the stem",
    "stem-width": "Width of the stem", "stem-root": "Type of stem root",
    "stem-surface": "Surface texture of the stem", "stem-color": "Color of the stem",
    "veil-type": "Type of veil", "veil-color": "Color of the veil",
    "has-ring": "Whether the mushroom has a ring", "ring-type": "Type of ring",
    "spore-print-color": "Color of the spore print", "habitat": "Where the mushroom grows",
    "season": "Season when observed", "class": "Edible or poisonous label",
    "has-stem": "Engineered: stem is present (yes/no)",
    "size-score": "Engineered: cap diameter + stem height + stem width / 10"
}

rows = []
for col in df.columns:
    numeric = pd.api.types.is_numeric_dtype(df[col])
    if numeric:
        observed = f"{df[col].min():.2f} to {df[col].max():.2f}"
    else:
        values = sorted(df[col].dropna().astype(str).unique())
        observed = ", ".join(values[:6]) + (" ..." if len(values) > 6 else "")
    rows.append({"Attribute": col, "Description": descriptions.get(col, col),
                 "Type": "Numeric" if numeric else "Categorical",
                 "Unique Values": df[col].nunique(),
                 "Observed Values / Range": observed,
                 "Missing": df[col].isna().sum()})

attribute_table = pd.DataFrame(rows)
display(attribute_table)

The original mushroom features include several missing categorical codes. Like Lab 2, we treat these as an explicit `Missing` category instead of trying to guess their values. We remove exact duplicates so identical observations are not given extra weight.

We also check for unusually large or small numerical values. These might be real observations rather than errors, so we do not automatically delete them.

In [ ]:
# Missing categories and potential numeric outliers
print("Number of 'Missing' values by column:")
missing_cat = (df == "Missing").sum().sort_values(ascending=False)
print(missing_cat[missing_cat > 0])

num_original = ["cap-diameter", "stem-height", "stem-width"]
outlier_rows = []
for col in num_original:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((df[col] < low) | (df[col] > high)).sum()
    outlier_rows.append({"Variable": col, "IQR Outlier Count": count,
                         "Percent": 100 * count / len(df)})

display(pd.DataFrame(outlier_rows).round(2))

### 2.4 Explore the Important Attributes

Before clustering, we look at the numerical measurements and some of the most common categories. This helps us understand how different the mushrooms are before applying any clustering algorithms.

In [ ]:
# Numerical distributions
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col in zip(axes, num_original):
    ax.hist(df[col], bins=35, edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel("Measurement")
    ax.set_ylabel("Mushrooms")
plt.tight_layout()
plt.show()

print(df[num_original].describe().round(2))

In [ ]:
# A few categorical distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["cap-shape", "habitat", "class"]):
    df[col].value_counts().head(8).plot(kind="bar", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("Mushrooms")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

**Initial observations:** The histograms show the spread of mushroom measurements, including possible extreme values. The bar charts show which cap shapes and habitats are common, along with the edible/poisonous balance for context. We will **not** put the edible/poisonous label into our clustering models.

**After running:** Add a brief observation here about which numerical variable has the widest spread and which cap shape or habitat appears most often.

## 3. Data Preparation (CRISP-DM)

### 3.1 Choose Variables for Clustering

Unlike classification, clustering does not have a target variable. We remove `class` and `does-bruise-or-bleed`, because these were our two Lab 2 targets and we want to examine them only after forming the clusters.

We also leave out `size-score` because it is calculated from the three numerical measurements already in the dataset. Including it would count the same size information twice. We leave out `has-stem` for a similar reason, since it is based on stem height and width.

This gives us the original morphological and environmental variables without the two labels or redundant engineered variables.

In [ ]:
# Predictors only: no target variables used for clustering
drop_cols = ["class", "does-bruise-or-bleed", "has-stem", "size-score"]
X = df.drop(columns=drop_cols)

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = [col for col in X.columns if col not in num_cols]

print("Observations:", len(X))
print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)
print("Total predictor columns:", X.shape[1])

### 3.2 Scale, Encode, and Reduce Dimensions

We use the same basic preprocessing methods as Lab 2:

- `StandardScaler` puts numerical variables on a similar scale.
- `OneHotEncoder` converts categorical variables into 0/1 columns. We do **not** assign ordered numeric codes to categories such as color.
- `PCA` reduces the resulting columns to 10 components so clustering runs more efficiently.

All clustering methods use this same representation so the comparisons are consistent. The first 10 principal components may not capture every detail, so we show their explained variance below. Euclidean distance is only an approximation of similarity for mixed categorical and numerical data; our encoding and weighting choices affect the clusters.

In [ ]:
# Same preprocessing structure as Lab 2
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])

X_encoded = preprocess.fit_transform(X)

# PCA keeps the cluster models reasonably fast
N_COMPONENTS = 10
pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_encoded)

print("After encoding:", X_encoded.shape)
print("After PCA:", X_pca.shape)
print("Variance explained:", round(pca.explained_variance_ratio_.sum() * 100, 2), "%")

In [ ]:
# PCA explained variance
plt.figure(figsize=(7, 4))
plt.plot(range(1, N_COMPONENTS + 1),
         pca.explained_variance_ratio_.cumsum() * 100, marker="o")
plt.title("Cumulative Variance Explained by PCA")
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Variance Explained (%)")
plt.xticks(range(1, N_COMPONENTS + 1))
plt.grid(alpha=0.3)
plt.show()

### 3.3 Choose a Modeling Sample

Our cleaned data has over 60,000 mushrooms. Agglomerative clustering is especially expensive on large datasets, so we use one random sample of up to **4,000 mushrooms** for choosing parameters and comparing all three methods. We use `random_state=42` so the sample can be reproduced.

K-Means can scale to the full dataset, so we will fit the selected K-Means configuration on **all** mushrooms later as a deployment example. The sample is a practical speed/coverage tradeoff, not a train/test split: no labels are used to train the clusters.

In [ ]:
# Same sample for all three methods
SAMPLE_SIZE = min(4000, len(df))
sample_idx = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).index.to_numpy()
X_sample = X_pca[sample_idx]
sample_df = df.iloc[sample_idx].reset_index(drop=True)

print("Clustering sample:", X_sample.shape)
print("Full data:", X_pca.shape)

# The known labels are kept separately for later interpretation
print("\nEdibility proportions (full dataset):")
print(df["class"].value_counts(normalize=True).round(3))
print("\nEdibility proportions (random sample):")
print(sample_df["class"].value_counts(normalize=True).round(3))

## 4. Modeling and Evaluation (CRISP-DM)

We compare **K-Means**, **Agglomerative Clustering**, and **DBSCAN**. They approach clustering differently:

- **K-Means** groups points around cluster centers. We must choose `k` (number of clusters).
- **Agglomerative Clustering** gradually combines nearby points. We use Ward linkage and also choose `k`.
- **DBSCAN** finds dense regions and labels some observations as noise. We select `eps` and `min_samples` instead of a fixed `k`.

### 4.1 Evaluation Measures

- **Silhouette score:** ranges roughly from -1 to 1; larger values usually mean better-separated groups.
- **Davies-Bouldin index:** measures how similar neighboring clusters are; smaller values are better.
- **Calinski-Harabasz score:** compares between-cluster and within-cluster dispersion; larger values are better for the same data representation.
- **Cluster sizes:** help us spot methods that put almost everything in one cluster or create many tiny groups.
- **DBSCAN noise percentage:** shows how many points DBSCAN could not assign to a dense group.

For DBSCAN, we compute the internal scores on **non-noise observations only**. Its scores therefore need to be interpreted together with the noise percentage and should not be compared with the other methods in isolation.

In [ ]:
# Function to evaluate a set of cluster labels
def cluster_scores(data, labels, exclude_noise=False):
    labels = np.asarray(labels)
    mask = labels != -1 if exclude_noise else np.ones(len(labels), dtype=bool)
    data_used = data[mask]
    labels_used = labels[mask]
    n_clusters = len(np.unique(labels_used))

    scores = {"Clusters": n_clusters,
              "Noise %": 100 * np.mean(labels == -1),
              "Silhouette": np.nan,
              "Davies-Bouldin": np.nan,
              "Calinski-Harabasz": np.nan}

    # These metrics need at least two nonempty clusters
    if 2 <= n_clusters < len(labels_used):
        try:
            scores["Silhouette"] = silhouette_score(
                data_used, labels_used,
                sample_size=min(1200, len(data_used)), random_state=RANDOM_STATE
            )
        except ValueError:
            # A small random sample can sometimes miss a tiny cluster
            scores["Silhouette"] = np.nan
        scores["Davies-Bouldin"] = davies_bouldin_score(data_used, labels_used)
        scores["Calinski-Harabasz"] = calinski_harabasz_score(data_used, labels_used)
    return scores

### 4.2 K-Means Clustering

We try several values of `k` and compare silhouette scores. We also create an elbow plot using inertia (the total within-cluster squared distance). We use `n_init=10` to reduce the effect of a bad starting point.

In [ ]:
# K-Means: try different numbers of clusters
kmeans_rows = []
kmeans_labels = {}

for k in range(2, 9):
    start_time = time.perf_counter()
    model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = model.fit_predict(X_sample)
    scores = cluster_scores(X_sample, labels)
    kmeans_rows.append({"k": k, "Inertia": model.inertia_,
                        **scores, "Time (s)": time.perf_counter() - start_time})
    kmeans_labels[k] = labels

kmeans_results = pd.DataFrame(kmeans_rows)
display(kmeans_results.round(3))

In [ ]:
# K-Means elbow and silhouette plots
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(kmeans_results["k"], kmeans_results["Inertia"], marker="o")
axes[0].set_title("K-Means: Elbow Plot")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia")

axes[1].plot(kmeans_results["k"], kmeans_results["Silhouette"], marker="o")
axes[1].set_title("K-Means: Silhouette Scores")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Silhouette")
plt.tight_layout()
plt.show()

# Choose k based on silhouette (without using edible/poisonous labels)
best_k = int(kmeans_results.loc[kmeans_results["Silhouette"].idxmax(), "k"])
best_kmeans_labels = kmeans_labels[best_k]
print("Selected K-Means k:", best_k)

**How to read these plots:** Inertia decreases as `k` increases, so we look for a point where the improvement starts to slow down. Silhouette is useful for comparing the separation of the solutions. We select the largest silhouette here as a reproducible starting choice, then check if the groups are sensible.

**After running:** Record the selected `k` and say whether the elbow plot supports a similar choice. If the plots disagree, discuss that rather than assuming one number must be correct.

### 4.3 Agglomerative (Hierarchical) Clustering

Next, we use Agglomerative Clustering with Ward linkage. Ward's method merges groups that lead to the smallest increase in within-cluster variation. We try several numbers of clusters and select the one with the best silhouette score on our common sample.

In [ ]:
# Agglomerative Clustering: choose the number of clusters
agg_rows = []
agg_labels = {}

for k in range(2, 9):
    start_time = time.perf_counter()
    model = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = model.fit_predict(X_sample)
    scores = cluster_scores(X_sample, labels)
    agg_rows.append({"k": k, **scores, "Time (s)": time.perf_counter() - start_time})
    agg_labels[k] = labels

agg_results = pd.DataFrame(agg_rows)
display(agg_results.round(3))

In [ ]:
# Agglomerative silhouette scores
plt.figure(figsize=(7, 4))
plt.plot(agg_results["k"], agg_results["Silhouette"], marker="o")
plt.title("Agglomerative Clustering: Silhouette Scores")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette")
plt.xticks(agg_results["k"])
plt.grid(alpha=0.3)
plt.show()

best_agg_k = int(agg_results.loc[agg_results["Silhouette"].idxmax(), "k"])
best_agg_labels = agg_labels[best_agg_k]
print("Selected Agglomerative k:", best_agg_k)

**Interpretation:** A high silhouette score suggests that Ward's method found groups that are compact and relatively well separated in the PCA space. We will compare its cluster sizes and actual mushroom characteristics with K-Means in Section 5.

**After running:** Add the selected `k` and whether the score is better or worse than the selected K-Means result.

### 4.4 DBSCAN

DBSCAN does not require a chosen number of clusters. Instead:

- `eps` controls how close observations need to be to count as neighbors.
- `min_samples` determines how many points are needed for a dense region.
- Observations that do not belong to any dense region receive label `-1` (noise).

We fix `min_samples=10` and use nearest-neighbor distances to choose a range of reasonable `eps` values. The number of clusters is an outcome of those settings, not something we specify beforehand.

In [ ]:
# Use 10th-nearest-neighbor distances to guide eps choices
MIN_SAMPLES = 10
neighbors = NearestNeighbors(n_neighbors=MIN_SAMPLES)
neighbors.fit(X_sample)
distances, _ = neighbors.kneighbors(X_sample)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 4))
plt.plot(k_distances)
plt.title("DBSCAN: 10th-Nearest-Neighbor Distances")
plt.xlabel("Observations (sorted)")
plt.ylabel("Distance")
plt.grid(alpha=0.3)
plt.show()

# Compare a range of eps values from the distance distribution
eps_values = sorted(set(np.round(np.quantile(k_distances,
                                    [0.50, 0.70, 0.85, 0.95, 0.99]), 3)))
eps_values = [float(eps) for eps in eps_values if eps > 0]
if not eps_values:
    eps_values = [0.5]  # fallback if many observations have identical features
print("eps values to test:", eps_values)

In [ ]:
# DBSCAN: compare eps values and the amount of noise
dbscan_rows = []
dbscan_labels = {}

for eps in eps_values:
    start_time = time.perf_counter()
    model = DBSCAN(eps=eps, min_samples=MIN_SAMPLES, n_jobs=-1)
    labels = model.fit_predict(X_sample)
    scores = cluster_scores(X_sample, labels, exclude_noise=True)
    dbscan_rows.append({"eps": eps, **scores,
                        "Time (s)": time.perf_counter() - start_time})
    dbscan_labels[eps] = labels

dbscan_results = pd.DataFrame(dbscan_rows)
display(dbscan_results.round(3))

In [ ]:
# Compare DBSCAN silhouette and noise percentage
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(dbscan_results["eps"], dbscan_results["Silhouette"], marker="o")
axes[0].set_title("DBSCAN: Silhouette (Non-Noise Points)")
axes[0].set_xlabel("eps")
axes[0].set_ylabel("Silhouette")

axes[1].plot(dbscan_results["eps"], dbscan_results["Noise %"], marker="o")
axes[1].set_title("DBSCAN: Percent Labeled Noise")
axes[1].set_xlabel("eps")
axes[1].set_ylabel("Noise (%)")
plt.tight_layout()
plt.show()

# Prefer a valid clustering with <= 50% noise, if one exists
candidates = dbscan_results[(dbscan_results["Clusters"] >= 2) &
                            (dbscan_results["Noise %"] <= 50) &
                            (dbscan_results["Silhouette"].notna())]
if candidates.empty:
    candidates = dbscan_results[(dbscan_results["Clusters"] >= 2) &
                                (dbscan_results["Silhouette"].notna())]

if candidates.empty:
    # DBSCAN may find only one group (or only noise). We still report this result.
    selected_row = dbscan_results.loc[dbscan_results["Noise %"].idxmin()]
else:
    selected_row = candidates.loc[candidates["Silhouette"].idxmax()]

best_eps = float(selected_row["eps"])
best_dbscan_labels = dbscan_labels[best_eps]
print("Selected DBSCAN eps:", best_eps)
print("Number of non-noise clusters:", int(selected_row["Clusters"]))
print("Percent labeled noise:", round(float(selected_row["Noise %"]), 2))

**How to read the DBSCAN results:** A small `eps` may leave many mushrooms labeled as noise. A large `eps` may merge most mushrooms into one cluster. We prefer an interpretable solution with more than one cluster and not too much noise, but we report the outcome even if DBSCAN does not work well with these features.

**After running:** State whether DBSCAN found meaningful groups, how many it found, and what percentage of the sample was considered noise. A high silhouette is less persuasive if most observations were discarded as noise.

## 5. Compare the Models and Interpret the Clusters

### 5.1 Overall Model Comparison

This table compares the selected setting for each method on the **same 4,000-mushroom sample**. The edible/poisonous labels are **still not used** in these internal scores.

For DBSCAN, scores exclude the observations labeled `-1`. That is why noise percentage and cluster sizes need to be considered alongside its silhouette score.

In [ ]:
# Store the selected clusters in one place
cluster_labels = {
    "K-Means": best_kmeans_labels,
    "Agglomerative": best_agg_labels,
    "DBSCAN": best_dbscan_labels
}

comparison_rows = []
for name, labels in cluster_labels.items():
    scores = cluster_scores(X_sample, labels, exclude_noise=(name == "DBSCAN"))
    comparison_rows.append({"Model": name, **scores})

comparison = pd.DataFrame(comparison_rows)
display(comparison.round(3))

### 5.2 Visualize the Clusters

We plot the first two PCA components to get a simple view of the groups. These plots are only a **two-dimensional projection**; the actual models use all 10 PCA components. A cluster may look overlapped here even if it is more separated in the full model space.

In [ ]:
# Visualize the selected clusters on the first two principal components
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, labels) in zip(axes, cluster_labels.items()):
    scatter = ax.scatter(X_sample[:, 0], X_sample[:, 1], c=labels,
                         cmap="tab20", s=9, alpha=0.65)
    ax.set_title(name)
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    fig.colorbar(scatter, ax=ax, label="Cluster")
plt.tight_layout()
plt.show()

### 5.3 Cluster Sizes

We also look at the number of mushrooms assigned to each cluster. Very small clusters may represent rare mushroom types or unusual observations. Extremely large clusters might mean the model is not finding much separation.

In [ ]:
# Cluster size tables and bar charts
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, labels) in zip(axes, cluster_labels.items()):
    sizes = pd.Series(labels).value_counts().sort_index()
    print("\n", name, "cluster sizes:")
    print(sizes.to_string())
    sizes.plot(kind="bar", ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Cluster (-1 = noise)")
    ax.set_ylabel("Mushrooms")
plt.tight_layout()
plt.show()

### 5.4 Describe the Mushroom Groups

The next tables help us understand what separates the clusters in the **original mushroom measurements**, not just the PCA values. We compare median cap diameter, stem height, and stem width, plus the most common cap shape, habitat, and gill spacing in each group.

For DBSCAN, cluster `-1` represents noise. If a method creates many clusters, we display the eight largest groups so the tables stay readable.

In [ ]:
# Describe the largest clusters using the original mushroom attributes
for name, labels in cluster_labels.items():
    profile = sample_df.copy()
    profile["Cluster"] = labels
    largest = profile["Cluster"].value_counts().head(8).index
    profile = profile[profile["Cluster"].isin(largest)]

    print("\n---", name, "---")
    print("Median numerical measurements:")
    display(profile.groupby("Cluster")[num_original].median().round(2))

    print("Most common category in each cluster:")
    top_categories = profile.groupby("Cluster")[["cap-shape", "habitat", "gill-spacing"]].agg(
        lambda s: s.value_counts().idxmax()
    )
    display(top_categories)

### 5.5 Compare Clusters with Known Edibility Labels

Now that the clusters have been created, we can use the original `class` variable as an **external interpretation check**. We did not use it in preprocessing, clustering, parameter selection, or the internal evaluation metrics.

A cluster containing mostly poisonous mushrooms might be interesting, but **clustering is not a substitute for a classifier**. We also calculate Adjusted Rand Index (ARI), which is approximately 0 for chance-level agreement and 1 for identical groupings. A low ARI does not necessarily mean the clusters have no value; edible/poisonous was not the objective used to fit them.

In [ ]:
# Post-hoc external check: compare cluster labels with edible/poisonous labels
ari_rows = []
for name, labels in cluster_labels.items():
    score = adjusted_rand_score(sample_df["class"], labels)
    ari_rows.append({"Model": name, "ARI vs. Edibility": score})

    print("\n---", name, "---")
    profile = sample_df.copy()
    profile["Cluster"] = labels
    edibility_pct = pd.crosstab(profile["Cluster"], profile["class"],
                                normalize="index") * 100
    display(edibility_pct.round(1))

print("\nAdjusted Rand Index comparison:")
display(pd.DataFrame(ari_rows).round(3))

**Interpretation:** The edibility table shows the percentage of edible and poisonous mushrooms **within** each cluster. This helps us see whether some groups are associated with edibility, but it does not establish that the groups are safe identification rules. For DBSCAN, ARI treats noise (`-1`) as an additional label, so interpret its score cautiously.

**After running, summarize:**

- Which method has the strongest separation based on silhouette and Davies-Bouldin?
- Does that method produce balanced, understandable groups, or a few very large clusters?
- Which physical characteristics differ most between its clusters?
- Do any clusters have noticeably different edibility proportions? What does the ARI suggest?

These answers are the main findings of the project. We should base them on the output tables rather than assume that one algorithm is always best.

## 6. Deployment (CRISP-DM)

Clustering could be useful as an exploratory tool for mushroom researchers or educators. For example, a user could enter physical measurements and categories and see which general group a mushroom resembles. Researchers could then look for patterns in those groups or study unusual observations.

**Limits:** We clustered on a random sample for our main comparison, and our feature representation uses Euclidean distance on scaled and one-hot-encoded data. Results could change if we used different feature weights, another region, or another set of mushrooms. Most importantly, cluster membership is **not** evidence that a mushroom is safe to eat.

**Possible deployment approach:** K-Means is the most straightforward method to apply to new mushrooms because it has a `predict` method. If its clusters are useful, we could refit it using all cleaned mushrooms and save the preprocessing, PCA, and K-Means steps together in a pipeline. Agglomerative Clustering and DBSCAN do not have a comparable built-in `predict` method in scikit-learn.

**How we would measure value:** Ask researchers whether cluster descriptions help identify patterns, check whether groups remain stable with newly collected data, and track changes in cluster sizes or feature distributions. We would also look for meaningful differences across regions or species if those labels become available.

**Additional data:** Confirmed mushroom species, location, expert annotations, weather, collection conditions, and more representative samples from new habitats. The model should be reviewed when these data change and retrained when the clusters no longer describe new observations well.

In [ ]:
# Example: fit a final K-Means pipeline on the entire dataset
# This is a deployment example, not a safety or edibility prediction model.
final_kmeans = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ])),
    ("pca", PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)),
    ("model", KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE))
])

final_kmeans.fit(X)
print("Final K-Means trained on", len(X), "mushrooms")
print("Example cluster assignments:", final_kmeans.predict(X.head(5)))

## 7. Exceptional Work: Check K-Means Stability

One additional check is to fit K-Means several times with different random seeds. We compare those cluster assignments using ARI. Here, ARI compares **one run of K-Means to another**, not to the edible/poisonous labels.

If the values are close to 1, K-Means is finding similar groups across initializations. If they are low, the exact cluster assignments depend more strongly on how the model starts.

In [ ]:
# Compare K-Means solutions with different random seeds
seeds = [7, 21, 42]
stability_labels = {}
for seed in seeds:
    model = KMeans(n_clusters=best_k, n_init=10, random_state=seed)
    stability_labels[seed] = model.fit_predict(X_sample)

stability_rows = []
for i, first_seed in enumerate(seeds):
    for second_seed in seeds[i + 1:]:
        stability_rows.append({
            "Seed 1": first_seed,
            "Seed 2": second_seed,
            "ARI Between Runs": adjusted_rand_score(
                stability_labels[first_seed], stability_labels[second_seed]
            )
        })

display(pd.DataFrame(stability_rows).round(3))

**After running:** Comment on whether the K-Means results were stable across the different seeds. High stability supports reproducibility, but it does not prove that the clusters are scientifically meaningful.

## 8. Final Conclusions

Our goal was to find meaningful groups in the mushroom dataset without using the known classification labels. We compared K-Means, Agglomerative Clustering, and DBSCAN using a common data representation and the same sampled observations. We evaluated the results using internal clustering scores, cluster sizes, visualizations, and the original mushroom characteristics.

**Complete this paragraph after running the notebook:** State which method you would choose, its selected parameters and key scores, the most interesting differences between its groups, and whether those groups showed any relationship to edibility. Also describe one limitation of the approach and how you would improve it with more data.

**Overall takeaway:** Clustering is useful for exploring patterns, but our results need external validation before any real-world use. It should not be used to determine mushroom safety.

### Reproducibility Notes

- Run the notebook from top to bottom in Python 3.
- Required libraries: `numpy`, `pandas`, `matplotlib`, `scipy`, `scikit-learn`, `ucimlrepo`, and `certifi`.
- The dataset is downloaded through `fetch_ucirepo(id=848)`, so internet access is needed the first time it is run.
- Cleaning functions are included here, exactly as in Lab 2 except for a pandas-compatible categorical-column check.
- The random sample and K-Means initializations use documented random seeds.
- If clustering is slow on your computer, reduce `SAMPLE_SIZE` (for example, 4000 to 2500) and rerun **all** comparisons. Mention that change in your report.
- Add specific interpretations to the marked Markdown sections after examining the output. Do not submit placeholders as final results.

**Source:** [UCI Secondary Mushroom Dataset](https://archive.ics.uci.edu/dataset/848/secondary+mushroom+dataset).